# Челленджи недели: функции, замыкания, декораторы, генераторы, with

**Цель:** проверить, что концепты недели работают вместе, а не каждый сам по себе.

**Как работать:**
- Часть задач сопровождается ячейкой «Объясни своими словами» — там нужно не только написать код, но и сформулировать, почему он работает.
- Все задачи решаются на 5-20 строк кода через `def`. Своих классов не пишем — `__enter__` / `__exit__` и пользовательские итераторы появятся на следующей неделе.
- Сначала попробуй сам, без подглядываний. Если застрял на 10+ минут — открой solution-версию.


## Задание 1: Декоратор-кэш для функции

Напиши декоратор `memoize`, который запоминает результаты вызовов функции в обычном словаре. При повторном вызове с теми же аргументами — возвращает запомненный результат, не пересчитывая.

Применишь его к рекурсивной `fib`. Без кэша `fib(35)` считается секунды; с кэшем — мгновенно.

Подсказки:
- кэш-словарь храни в замыкании
- ключом служит кортеж `args` (он хэшируемый)
- не забудь `@functools.wraps`

In [32]:
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    def __init__(self, name, breed):
        self.breed = breed

d = Dog("Rex", "corgi")
print(d.breed)
print(d.name)


corgi


AttributeError: 'Dog' object has no attribute 'name'

In [9]:
from functools import wraps
import time

# TODO: implement
def memorize(func):
    memory = {}
    @wraps(func)
    def wrapper(*args):
        if args not in memory:
            memory[args] = func(*args)
        return memory[args]
    return wrapper

@memorize
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)


start_time = time.time()
print(fibonacci(10))
end_time = time.time()
print(f"Time taken: {end_time - start_time}")

start_time = time.time()
print(fibonacci(10))
end_time = time.time()
print(f"Time taken: {end_time - start_time}")

55
Time taken: 7.677078247070312e-05
55
Time taken: 2.5033950805664062e-05


**Объясни своими словами:** где живёт словарь `cache` после того, как `memoize` вернул `wrapper`, и почему он не сбрасывается между вызовами?

Он живет в замыкании функции "wrapper". И когда мы вызываем "wrapper", он обращается к словарю "memory" в замыкании, который сохраняет свои значения между вызовами.

## Задание 2: Генератор чисел Фибоначчи

Напиши генератор `fib_up_to(limit)`, который выдаёт числа Фибоначчи **меньше** `limit`. Последовательность начинается с `0, 1, 1, 2, 3, 5, 8, ...`.

Используй `yield` в `while`-цикле, без хранения всей последовательности в памяти.

Проверка: `list(fib_up_to(50))` должно вернуть `[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]`.

In [ ]:

# TODO: implement
# your code:
def fib_up_to(limit):
    a, b = 0, 1
    while a < limit:
        yield a
        a, b = b, a+b

list(fib_up_to(50))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## Задание 3: Контекстный менеджер для замера времени

Напиши контекстный менеджер `timed(label)` через `@contextlib.contextmanager`. Он замеряет, сколько прошло времени между входом и выходом из блока, и печатает результат.

Требования:
- замер работает даже при исключении внутри блока (используй `try/finally`)
- формат вывода: `[<label>] N.NNNN сек`

Проверь на `time.sleep(0.1)`.

In [14]:
from contextlib import contextmanager
import time

# TODO: implement
# your code:
@contextmanager 
def timer(label):
    start_time = time.time()
    try:
        yield
    finally:
        end_time = time.time()
        print(f"[{label}] {end_time - start_time:.4f} сек")

with timer("sleep 1 sec"):
    time.sleep(1)

with timer("fibonacci(35)"):
    fibonacci(35)

[sleep 1 sec] 1.0051 сек
[fibonacci(35)] 0.0000 сек


**Объясни своими словами:** что произойдёт, если убрать `try/finally` и оставить только `yield` в теле менеджера?

при исключении внутри блока, код после `yield` не выполнится, и замер времени не будет напечатан.

## Задание 4: Подводный камень с изменяемым дефолтом

Дана «наивная» функция `add_item`, у которой дефолт-параметр `basket=[]`. Покажи, что несколько вызовов без явного `basket` делят один и тот же список (это и есть mutable default trap).

Затем напиши **исправленную** версию `add_item_safe`, которая ведёт себя ожидаемо: каждый вызов без `basket` получает свежий пустой список.

Стандартный шаблон починки — `basket=None` + проверка внутри функции.

In [16]:

# TODO: implement
# your code:
def add_item(item, basket=[]):
    basket.append(item)
    return basket

def add_item_safe(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item('apple'))  # ['apple']
print(add_item('banana'))  # ['apple', 'banana']

print(add_item_safe('apple'))  # ['apple']
print(add_item_safe('banana'))  # ['banana']

['apple']
['apple', 'banana']
['apple']
['banana']


**Объясни своими словами:** почему именно `None` как дефолт + проверка `if basket is None` — стандартный паттерн, а не, например, `basket=list()`?

Потому что при None мы создаем новый объект только тогда, когда он действительно нужен, а не при каждом определении функции. Если использовать `basket=list()`, то список будет создан один раз при определении функции и будет общим для всех вызовов без аргумента.

## Задание 5: Фабрика счётчиков через замыкание

Напиши функцию-фабрику `make_counter(start=0, step=1)`, которая возвращает функцию-счётчик. Каждый вызов счётчика увеличивает его значение на `step` и возвращает текущее значение.

Состояние храни в замыкании. Помни: чтобы менять переменную из внешней области в Python, нужен `nonlocal`.

Проверка:
```
c = make_counter(start=10, step=5)
c()  # 15
c()  # 20
c()  # 25
```

In [19]:

# TODO: implement
# your code:
def make_counter(start=0, step=1):
    counter = start
    def count():
        nonlocal counter
        counter += step
        return counter
    return count

c = make_counter(start=10, step=5)
print(c())  # 15
print(c())  # 20
print(c())  # 25

15
20
25


## Задание 6: Генераторное выражение для фильтрации логов

Дан список лог-строк. Напиши **одно** выражение, которое:

- оставляет только строки, начинающиеся с `ERROR`
- из каждой такой строки извлекает только сообщение (часть после `ERROR: `)

Используй генераторное выражение. Передай его в `list(...)`, чтобы материализовать результат.

In [20]:
log_lines = [
    "INFO: server started",
    "ERROR: db connection failed",
    "WARN: slow query",
    "ERROR: timeout after 30s",
    "INFO: request handled",
    "ERROR: invalid token",
]

# TODO: implement
# your code:
errors = (line[7:] for line in log_lines if line.startswith('ERROR'))
print(list(errors))

['db connection failed', 'timeout after 30s', 'invalid token']


## Задание 7: Декоратор с аргументами — `retry`

Напиши декоратор `retry(times)`, который при падении функции с исключением — повторяет её до `times` раз. Если все попытки провалились — пробрасывает последнее исключение наружу.

Структура — три уровня вложенности: внешняя функция принимает `times`, возвращает декоратор, который возвращает `wrapper`.

Подсказка для проверки: внутри тестовой функции держим счётчик падений в замыкании и роняем её на первых двух вызовах, чтобы увидеть retry в действии.

In [25]:
from functools import wraps

# TODO: implement
# your code:
def retry(times):
    def dec(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
            raise last_exception
        return wrapper
    return dec

retry_3 = retry(3)

@retry_3
def unreliable_function():
    import random
    if random.random() < 0.7:
        raise ValueError("Random failure!")
    return "Success!"

print(unreliable_function())

ValueError: Random failure!

**Объясни своими словами:** зачем нужен **третий** уровень вложенности (внешняя функция, декоратор, wrapper), и что бы случилось, если попробовать обойтись двумя?

Он нужен для того, чтобы создавать разные декораторы путем передачи аргумента. Если использовать количество повторов в аргументах decor, то мы не сможем передавать разные значения при каждом вызове декоратора. Если обойтись двумя уровнями, то мы не сможем передавать параметр `times` в декоратор и он будет фиксированным для всех функций, которые мы хотим декорировать.

## Задание 8: Сортировка словарей по нескольким полям

Дан список словарей с информацией о сотрудниках. Отсортируй его по двум полям одновременно: сначала по убыванию `salary`, затем (при равной зарплате) по возрастанию `name`.

Используй `sorted(...)` с параметром `key=` и `lambda`. Подсказка: вернуть из лямбды кортеж — `sorted` сравнит кортежи поэлементно. Чтобы отсортировать поле по убыванию — поставь перед ним знак `-` (для чисел) или используй параметр `reverse=` (сложнее, когда направления разные).

In [27]:
employees = [
    {"name": "Аня",   "salary": 100_000},
    {"name": "Боря",  "salary": 120_000},
    {"name": "Вера",  "salary": 100_000},
    {"name": "Гена",  "salary": 80_000},
    {"name": "Дима",  "salary": 120_000},
]

# TODO: implement
# your code:
employees_sorted = sorted(employees, key=lambda x: (-x['salary'], x['name']))
employees_sorted

[{'name': 'Боря', 'salary': 120000},
 {'name': 'Дима', 'salary': 120000},
 {'name': 'Аня', 'salary': 100000},
 {'name': 'Вера', 'salary': 100000},
 {'name': 'Гена', 'salary': 80000}]

# Готово

Ты только что прошёл задачи на пересечении функций, замыканий, декораторов, генераторов и контекстных менеджеров. Если все ячейки прошли — концепты недели у тебя работают как единое целое.

На следующей неделе мы разберём ООП — и увидим, как замыкания и декораторы из этой недели превращаются в классы с состоянием, а пользовательские итераторы и контекстные менеджеры пишутся через магические методы.
